# Jumper: a GPU training session in Google Colab

Choose **Runtime → Change runtime type → GPU**, then run the cells in order. A paid runtime is not required by this notebook; GPU availability, model and session duration are decided by Colab. No GPU means a clear stop, never a silent CPU training run.

The session uses the repository's existing training, replay and ONNX export commands in an isolated `.venv`. The Colab kernel only coordinates subprocesses and displays results. First watch a bundled pretrained dance, then run a five-iteration smoke test and train a modest walking run. A short training run is a workflow check, not a promise of a learned gait or a hardware-ready robot.

Optional Drive snapshots happen **during training**, each time a complete checkpoint is saved. Without Drive, download the final backup before Colab discards `/content`. Snapshots retain a checkpoint with its source/runtime manifest; writes still require time and an abrupt VM loss can interrupt an unfinished copy.

This notebook checks the **training runtime**. It does not set up Rust, RKNN, DDS or the physical motor bus. An ONNX export is not a robot deployment bundle.

## 1. Session choices

The defaults select the accompanying release branch in the tianrking/jumper fork. After the change is merged, you can select KingKongRobotics/jumper and main instead. When testing another fork or release branch, enter its exact repository URL and revision. The requested revision is resolved to one commit and recorded; an existing checkout is reused only if its origin and current commit agree. It is never reset or replaced.

The source revision must contain `mjrl.colab` and the replay video options. Setup stops early if you selected older code. For later reproducible sessions, paste the printed 40-character commit into `SOURCE_REVISION`.

Select a single explicit saved checkpoint when resuming. Restore a downloaded ZIP in section 5, or enter a checkpoint inside a mounted Drive snapshot. Resume refuses changes to the task, model, backend, number of environments, source content or core runtime package versions.

In [ ]:
REPOSITORY_URL = "https://github.com/tianrking/jumper.git"  # @param {type:"string"}
SOURCE_REVISION = "codex/colab-training-and-video"  # @param {type:"string"}
TASK = "jumper.tripod"  # @param ["jumper.flat", "jumper.tripod", "jumper.tetrapod", "jumper.ripple", "jumper.posture", "jumper.five_foot", "jumper.dance", "jumper.jump", "jumper.ref_free_jump", "jumper.swing"]
MODEL = "jumper"
NUM_ENVS = 256  # @param {type:"integer"}
ITERATIONS = 500  # @param {type:"integer"}
USE_DRIVE = False  # @param {type:"boolean"}
UPLOAD_BACKUP = False  # @param {type:"boolean"}
RESUME_CHECKPOINT = ""  # @param {type:"string"}
DRIVE_FOLDER = "JumperColab"  # @param {type:"string"}

if NUM_ENVS < 1 or ITERATIONS < 1:
    raise ValueError("NUM_ENVS and ITERATIONS must be positive")
if not SOURCE_REVISION.strip() or SOURCE_REVISION.startswith("-"):
    raise ValueError("Enter a branch, tag or commit as SOURCE_REVISION")

## 2. Resolve the checkout and create an isolated interpreter

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import uuid
import venv

if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError("Jumper supports Python 3.10–3.13; select a compatible Colab runtime")
if not Path("/content").is_dir():
    raise RuntimeError("This notebook is intended for Google Colab with /content available")

def run(command, *, cwd=None):
    print("+", " ".join(str(part) for part in command), flush=True)
    return subprocess.run([str(part) for part in command], cwd=cwd, check=True)

# Fail before downloads when no GPU was attached to this runtime.
run(["nvidia-smi", "-L"])
gpu_rows = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"],
    text=True,
).strip().splitlines()
if not gpu_rows:
    raise RuntimeError("No GPU detected; change the Colab runtime type to GPU")
gpu_name, capability = [part.strip() for part in gpu_rows[0].rsplit(",", 1)]
capability = float(capability)
if capability < 7.5:
    raise RuntimeError(f"Unsupported GPU for this notebook's tested wheel family: {gpu_name}")
print("GPU:", gpu_name, "compute capability:", capability)

REPO = Path("/content/jumper")
created_checkout = not REPO.exists()
if created_checkout:
    run(["git", "clone", REPOSITORY_URL, REPO])
else:
    if not (REPO / ".git").exists():
        raise RuntimeError(f"{REPO} already exists and is not a Git checkout")
    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO, text=True,
    ).strip()
    if origin.rstrip("/").removesuffix(".git") != REPOSITORY_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Existing checkout origin differs: {origin}")

# Fetching updates references; no checkout content is overwritten.
run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO)
COMMIT = subprocess.check_output(
    ["git", "rev-parse", "FETCH_HEAD^{commit}"], cwd=REPO, text=True,
).strip()
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
if head != COMMIT:
    # A freshly cloned clean checkout may be pinned. Reused checkouts are never switched.
    status = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True)
    if not created_checkout or status:
        raise RuntimeError(f"Existing checkout is {head}, requested {COMMIT}; use a new runtime")
    run(["git", "checkout", "--detach", COMMIT], cwd=REPO)
if not (REPO / "rl/mjrl/colab.py").is_file():
    raise RuntimeError(
        f"Revision {COMMIT} does not contain this notebook's Colab helper. "
        "Choose the accompanying release/fork revision, then rerun in a fresh runtime."
    )
print("Pinned source commit:", COMMIT)

PYTHON = REPO / ".venv/bin/python"
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=True, system_site_packages=False).create(REPO / ".venv")
run([PYTHON, "-c", "import sys; print('isolated interpreter:', sys.executable, sys.prefix)"])
SESSION = Path("/content/jumper-colab") / uuid.uuid4().hex
SESSION.mkdir(parents=True)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONUNBUFFERED"] = "1"

## 3. Install and verify the training runtime

Dependencies are installed only into this checkout's `.venv`. The notebook pins the paired PyTorch 2.9.1 / torchvision 0.24.1 wheels from the [official PyTorch archive](https://pytorch.org/get-started/previous-versions/): CUDA 12.6 for a T4, CUDA 12.8 for Ampere/Ada/Blackwell. A real CUDA operation and a compiled Warp CUDA kernel must succeed; package installation alone is not acceptance.

Other dependencies follow `pyproject.toml`; the exact core versions are saved for resume checks. A driver or wheel mismatch raises an error. Do not continue past a failed gate.

In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "ffmpeg"])
run([PYTHON, "-m", "pip", "install", "--upgrade", "pip"])
TORCH_INDEX = "cu126" if capability < 8.0 else "cu128"
run([
    PYTHON, "-m", "pip", "install", "torch==2.9.1", "torchvision==0.24.1",
    "--index-url", f"https://download.pytorch.org/whl/{TORCH_INDEX}",
])
run([PYTHON, "-m", "pip", "install", "-e", str(REPO)])
run([PYTHON, "-m", "pip", "check"])
run([
    PYTHON, "-m", "mjrl.colab", "check", "--repo", REPO,
    "--out", SESSION / "runtime.json",
], cwd=REPO)
runtime = json.loads((SESSION / "runtime.json").read_text())
print(json.dumps(runtime, indent=2))
run([PYTHON, "scripts/train.py", "--list"], cwd=REPO)

# Detect an older replay CLI before attempting to train anything.
play_help = subprocess.check_output(
    [str(PYTHON), "scripts/play.py", "--help"], cwd=REPO, text=True,
)
required_video_flags = ("--video", "--video-fps", "--video-width", "--video-height")
if any(flag not in play_help for flag in required_video_flags):
    raise RuntimeError("This source revision lacks the notebook's replay video interface")

## 4. Optional Drive persistence

In [ ]:
PERSIST_ROOT = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if not DRIVE_FOLDER or Path(DRIVE_FOLDER).is_absolute() or ".." in Path(DRIVE_FOLDER).parts:
        raise ValueError("DRIVE_FOLDER must be a relative folder under MyDrive")
    PERSIST_ROOT = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
    print("Complete checkpoint snapshots will be saved under:", PERSIST_ROOT)
else:
    print("Drive is off. Results remain in /content until you download them.")

## 5. Watch the bundled pretrained dance; optionally restore your backup

The dance checkpoint is included in the repository. The video is a finite ten-second policy replay at the dance task's training physics rate (200 Hz), with training observation noise enabled. It demonstrates an existing checkpoint, not the untrained walking policy.

The optional upload restores a notebook backup into a **new** directory. It rejects traversal paths, symlinks, duplicate entries, oversized archives and replacement of existing runs. It selects a checkpoint only inside that restored run; compatibility is checked before training.

In [ ]:
from IPython.display import Video, display

DEMO = REPO / "tasks/jumper/dance/out/example/model_2300.pt"
if not DEMO.is_file():
    raise FileNotFoundError(f"Bundled pretrained demo checkpoint is missing: {DEMO}")
DEMO_VIDEO = SESSION / "pretrained-dance.mp4"
run([
    PYTHON, "scripts/play.py", "--task", "jumper.dance", "--model", MODEL,
    "--checkpoint", DEMO, "--backend", "warp", "--device", "cuda:0",
    "--num_envs", "1", "--headless", "--steps", "500", "--physics-hz", "200",
    "--video", DEMO_VIDEO, "--video-fps", "30",
    "--video-width", "640", "--video-height", "480", "--strip-visual", "off",
], cwd=REPO)
if not DEMO_VIDEO.is_file() or DEMO_VIDEO.stat().st_size == 0:
    raise RuntimeError("Replay did not produce the demo MP4")
display(Video(str(DEMO_VIDEO), embed=True))

In [ ]:
if UPLOAD_BACKUP:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one notebook backup ZIP")
    original_name, payload = next(iter(uploaded.items()))
    archive = SESSION / "uploaded-backup.zip"
    archive.write_bytes(payload)
    restored = SESSION / "restored"
    restore_info = SESSION / "restore.json"
    run([
        PYTHON, "-m", "mjrl.colab", "restore", "--archive", archive,
        "--destination", restored, "--out", restore_info,
    ], cwd=REPO)
    RESUME_CHECKPOINT = json.loads(restore_info.read_text())["checkpoint"]
    print("Explicit restored resume checkpoint:", RESUME_CHECKPOINT)
else:
    print("No ZIP upload requested.")
if RESUME_CHECKPOINT:
    if not Path(RESUME_CHECKPOINT).is_file():
        raise FileNotFoundError(f"Resume checkpoint does not exist: {RESUME_CHECKPOINT}")
    print("Selected resume checkpoint:", RESUME_CHECKPOINT)

## 6. Five-iteration GPU smoke test

This uses 256 environments, `warp` and `cuda:0` explicitly. It must produce a new log directory and a complete checkpoint. If the assigned GPU runs out of memory, stop and reduce the configured batch for a new training session; this cell does not disguise an error as CPU success.

The smoke run is separate from the chosen training run and is never implicitly selected for resume.

In [ ]:
SMOKE_MANIFEST = SESSION / "smoke.json"
run([
    PYTHON, "-m", "mjrl.colab", "train", "--repo", REPO,
    "--task", "jumper.tripod", "--model", MODEL, "--num-envs", "256",
    "--iterations", "5", "--manifest", SMOKE_MANIFEST,
], cwd=REPO)
smoke = json.loads(SMOKE_MANIFEST.read_text())
if smoke["status"] != "finished":
    raise RuntimeError(f"Smoke test failed: {smoke}")
print("GPU smoke test completed:", smoke["run_directory"])

## 7. Train the selected task

`ITERATIONS` is the number of updates **this run performs**; on resume these are additional updates. The default 500 is deliberately modest and is not a convergence criterion. The repository's policies may require thousands of updates. Training with 256 environments changes PPO batch size from the original large GPU runs; matching their learning curves requires separate measurement.

Every complete new checkpoint is CRC-checked, hashed and recorded. With Drive on, the checkpoint and its manifest are copied to a temporary folder, then published together while training continues. A failed subprocess or interrupted copy remains an error, with a local failure manifest for diagnosis. Saved checkpoints remain usable even if later training fails.

Keep the same task, model, environment count and source/runtime versions when resuming. A bundled example checkpoint has no notebook run manifest and is suitable for replay, not implicit notebook resume.

In [ ]:
TRAIN_MANIFEST = SESSION / "training.json"
train_command = [
    PYTHON, "-m", "mjrl.colab", "train", "--repo", REPO,
    "--task", TASK, "--model", MODEL, "--num-envs", str(NUM_ENVS),
    "--iterations", str(ITERATIONS), "--manifest", TRAIN_MANIFEST,
]
if PERSIST_ROOT is not None:
    train_command += ["--drive-root", PERSIST_ROOT]
if RESUME_CHECKPOINT:
    train_command += ["--resume", Path(RESUME_CHECKPOINT)]
run(train_command, cwd=REPO)
training = json.loads(TRAIN_MANIFEST.read_text())
if training["status"] != "finished":
    raise RuntimeError(f"Training did not finish: {training}")
RUN_DIR = Path(training["run_directory"])
CHECKPOINT_PATH = Path(training["checkpoint"])
print("This run:", RUN_DIR)
print("Selected checkpoint:", CHECKPOINT_PATH)

## 8. Replay and download your trained policy video

The task's own physics rate is queried from its configuration in the isolated process and passed explicitly. The video keeps the task's control rate, uses one environment and ends after 500 control steps. Observation noise stays enabled. This is a policy preview; examine tracking errors and repeated runs before interpreting it as acceptance.

In [ ]:
rates_file = SESSION / "rates.json"
run([
    PYTHON, "-m", "mjrl.colab", "rates", "--repo", REPO,
    "--task", TASK, "--model", MODEL, "--out", rates_file,
], cwd=REPO)
rates = json.loads(rates_file.read_text())
print("Replay rates:", rates)
TRAIN_VIDEO = SESSION / "trained-policy.mp4"
run([
    PYTHON, "scripts/play.py", "--task", TASK, "--model", MODEL,
    "--checkpoint", CHECKPOINT_PATH, "--backend", "warp", "--device", "cuda:0",
    "--num_envs", "1", "--headless", "--steps", "500",
    "--physics-hz", str(rates["physics_hz"]), "--video", TRAIN_VIDEO,
    "--video-fps", "30", "--video-width", "640", "--video-height", "480",
    "--strip-visual", "off",
], cwd=REPO)
if not TRAIN_VIDEO.is_file() or TRAIN_VIDEO.stat().st_size == 0:
    raise RuntimeError("Replay did not produce the trained MP4")
display(Video(str(TRAIN_VIDEO), embed=True))

In [ ]:
from google.colab import files
files.download(str(TRAIN_VIDEO))

## 9. Export ONNX and download a portable backup

Export uses the same repository entry point and explicitly selected checkpoint. Its normal contract and ONNX checks run; `--no-video` avoids separately rendering a task's full motion clip. No RKNN conversion, board bundle or physical deployment happens here.

The ZIP includes this run's checkpoints, parameter/log files, provenance manifest, ONNX export and preview MP4. Restore it through section 5 in a later session, select the recorded source commit and matching runtime, and continue with the restored explicit checkpoint.

If rendering or export fails, you can still run the backup cell below: it includes those artifacts only when they exist. Drive checkpoint snapshots continue to be the protection during training; a final ZIP alone cannot protect an interrupted session.

In [ ]:
EXPORT_DIR = SESSION / "onnx-export"
if EXPORT_DIR.exists():
    raise FileExistsError("Choose a fresh export directory before rerunning this cell")
run([
    PYTHON, "scripts/export.py", "--task", TASK, "--model", MODEL,
    "--checkpoint", CHECKPOINT_PATH, "--backend", "warp", "--device", "cuda:0",
    "--num_envs", "1", "--no-video", "--out", EXPORT_DIR,
], cwd=REPO)
if not (EXPORT_DIR / "actor.onnx").is_file() or not (EXPORT_DIR / "layout.json").is_file():
    raise RuntimeError("ONNX export did not produce its policy and contract")
print("ONNX policy and contract:", EXPORT_DIR)

In [ ]:
# The training manifest is the source of truth even if a later preview/export cell failed.
training = json.loads(TRAIN_MANIFEST.read_text())
if not training.get("run_directory") or not training.get("checkpoints"):
    raise RuntimeError("No complete training checkpoint is available to back up")
RUN_DIR = Path(training["run_directory"])
BACKUP_ZIP = SESSION / f"jumper-{TASK.replace('.', '-')}-{uuid.uuid4().hex[:8]}.zip"
backup_command = [
    PYTHON, "-m", "mjrl.colab", "backup", "--run", RUN_DIR, "--out", BACKUP_ZIP,
]
candidate_export = SESSION / "onnx-export"
if (candidate_export / "actor.onnx").is_file() and (candidate_export / "layout.json").is_file():
    backup_command += ["--export", candidate_export]
candidate_video = SESSION / "trained-policy.mp4"
if candidate_video.is_file() and candidate_video.stat().st_size:
    backup_command += ["--video", candidate_video]
run(backup_command, cwd=REPO)
print("Portable backup:", BACKUP_ZIP, "bytes:", BACKUP_ZIP.stat().st_size)
files.download(str(BACKUP_ZIP))

## Interruption and troubleshooting

- **No CUDA / Warp kernel failure:** attach a GPU runtime, inspect the driver and wheel errors, and rerun setup. No automatic CPU fallback is enabled.
- **Out of memory:** use a smaller training batch in a new run. Resume requires the recorded batch; changing it is a new experiment.
- **Task or source mismatch:** restore the same source commit and core package versions, not merely a checkpoint with a similar filename.
- **Training failed:** inspect `training.json` and console output. Download the ZIP of any complete saved checkpoints, or resume from a published Drive snapshot's explicit `model_*.pt` after a matching setup.
- **A Colab runtime disconnects:** inspect Drive snapshot folders; folders named `.pending-*` were not published and should not be selected. Without Drive or a downloaded backup, `/content` results can be lost.
- **Render/export failed:** the training run and checkpoint manifest remain available. The backup cell accepts a run without successful video or ONNX output.

The notebook's static checks and local process/backup tests are distinct from an actual Colab GPU session. Do not infer that a particular Colab GPU, training outcome or robot has been validated from the existence of this notebook.